# Explore the PBMC SLE pipeline step by step (Colab)

Same code as the pipeline scripts, imported as functions and run one cell at a time, so you can look at the data
between steps and tune settings. `train_colab.ipynb` stays the "Run all" path for the final model.

- **Knobs** are in the `P` cell. Change one, re-run from the step it affects.
- **Edit the code:** open a file in the left Files panel (e.g. `immune-research/projects/pbmc_sle_classification/train.py`),
  change it, save (Ctrl+S). `%autoreload` picks it up on the next cell run. Or copy a function into a cell and rewrite it there.
- **Cache:** the QC'd data, the scVI model and its output go to Google Drive, so the 11 GB download and scVI run once.
  Delete a cached file (or set `REBUILD`) to recompute that step.
- When a setting wins, put it in `configs/default.yaml` and commit (the last cell prints the YAML).

Runtime: T4 GPU, High-RAM if you raise `max_cells`.

In [ ]:
# 1 · setup: clone the repo, install packages (restarts once), mount Drive
BRANCH = "main"
import os, sys, subprocess
IN_COLAB = os.path.exists("/content")
os.chdir("/content" if IN_COLAB else os.getcwd())
if IN_COLAB and not os.path.exists("immune-research"):
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/adamsi/immune-research.git"], check=True)
if os.path.exists("immune-research"):
    os.chdir("immune-research")
sys.path.insert(0, os.getcwd())   # import the pipeline as `projects.pbmc_sle_classification`
if IN_COLAB and not os.path.exists(".deps_installed"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "projects/pbmc_sle_classification/requirements.txt"], check=True)
    open(".deps_installed", "w").close()
    print("Installed. Restarting the runtime now. When it reconnects, run this cell again.")
    os.kill(os.getpid(), 9)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/pbmc_cache" if IN_COLAB else "pbmc_cache"
os.makedirs(CACHE, exist_ok=True)
%load_ext autoreload
%autoreload 2
print("repo:", os.getcwd(), "| cache:", CACHE)

In [ ]:
# 2 · knobs (re-run the steps below the one you change)
P = dict(
    max_cells=500,        # cells per sample (data step)
    n_top_genes=5000,     # HVGs (HVG step)
    max_pct_mt=15,        # QC
    n_latent=30, n_layers=1, scvi_epochs=None,   # scVI (None = scvi-tools default)
    k="auto",             # number of SLE subtypes, or an int
    n_heads=4, d=128, epochs=40, lr=1e-3, n_sub=500, weight_decay=0.01,   # MIL
    baseline_C=0.1, baseline_l1_ratio=0.5,       # elastic-net baseline
)
REBUILD = set()           # e.g. {"data", "scvi"} to ignore the cache for those steps

import json, yaml, numpy as np, pandas as pd, scanpy as sc, anndata as ad, matplotlib.pyplot as plt
from projects.pbmc_sle_classification import DEFAULT_CONFIG, path
cfg = yaml.safe_load(open(DEFAULT_CONFIG))
cfg["data"]["max_cells_per_sample"] = P["max_cells"]
cfg["qc"].update(n_top_genes=P["n_top_genes"], max_pct_mt=P["max_pct_mt"])
cfg["represent"].update(n_latent=P["n_latent"], n_layers=P["n_layers"], scvi_epochs=P["scvi_epochs"])
cfg["endotypes"]["k"] = P["k"]
cfg["model"].update(n_heads=P["n_heads"], d=P["d"])
cfg["train"].update({k: P[k] for k in ("epochs", "lr", "n_sub", "weight_decay", "baseline_C", "baseline_l1_ratio")})
D = cfg["data"]
cached = lambda name: os.path.join(CACHE, name)
use_cache = lambda name, step: os.path.exists(cached(name)) and step not in REBUILD
sc.settings.set_figure_params(dpi=90, frameon=False)

## 3 · Data: download, subsample, QC (incl. platelet filter), Perez labels

In [ ]:
from projects.pbmc_sle_classification.data import download, load_subsample, pick_batch_key, qc, fine_labels, batch_report

if use_cache("pbmc_qc.h5ad", "data"):
    adata = ad.read_h5ad(cached("pbmc_qc.h5ad"))
else:
    src = download(cfg)                                   # 11 GB, once per runtime (local disk, not Drive)
    adata = load_subsample(src, D["max_cells_per_sample"], cfg["train"]["seed"], D["sample_key"])
    adata.uns["batch_key"] = pick_batch_key(adata.obs, D["batch_key"]) or ""
    adata = qc(adata, cfg["qc"])
    adata.obs["cell_type_fine"] = fine_labels(adata.obs, cfg["labels"]["fine"])
    adata.obs["lineage"] = adata.obs["author_cell_type"].astype(str)
    adata.write_h5ad(cached("pbmc_qc.h5ad"))
BATCH = adata.uns["batch_key"] or None
print(adata)
display(batch_report(adata.obs, BATCH, D["sample_key"]))
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], groupby="disease_state", rotation=30)

In [ ]:
# Perez cell labels: how many cells of each type, per disease state
display(pd.crosstab(adata.obs["cell_type_fine"], adata.obs["disease_state"], normalize="columns").mul(100).round(1))

## 4 · Normalise + HVGs

In [ ]:
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=cfg["qc"]["n_top_genes"], flavor="seurat_v3", layer="counts", batch_key=BATCH)
print(adata.var["highly_variable"].sum(), "HVGs")
sc.pl.highly_variable_genes(adata)

## 5 · scVI latent space + UMAP

In [ ]:
from projects.pbmc_sle_classification.represent import train_scvi
import scvi

if use_cache("scvi_latent.npy", "scvi") and np.load(cached("scvi_latent.npy")).shape[0] == adata.n_obs:
    adata.obsm["X_scVI"] = np.load(cached("scvi_latent.npy"))
else:
    vae, genes = train_scvi(adata, BATCH, cfg["represent"])
    adata.obsm["X_scVI"] = vae.get_latent_representation()
    np.save(cached("scvi_latent.npy"), adata.obsm["X_scVI"])
    vae.save(cached("scvi_model"), overwrite=True)
    plt.plot(vae.history["elbo_train"]); plt.title("scVI training ELBO"); plt.show()
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["cell_type_fine", "disease_state", BATCH], ncols=1, size=3, legend_fontsize=7)

## 6 · Cell abundance

In [ ]:
from projects.pbmc_sle_classification.features import sample_table
from projects.pbmc_sle_classification.insights import composition_test

samples = sample_table(adata.obs, D["sample_key"], D["donor_key"], BATCH, D["sle_label"])
comp = composition_test(adata, samples, D["sample_key"])
display(comp.head(12))
frac = pd.crosstab(adata.obs[D["sample_key"]].astype(str), adata.obs["cell_type_fine"], normalize="index") * 100
state = samples["disease_state"].reindex(frac.index)
top = comp.index[:6]
fig, axes = plt.subplots(1, len(top), figsize=(3 * len(top), 3.2))
for ax, ct in zip(axes, top):
    states = sorted(state.unique())
    ax.boxplot([frac[ct][state == s] for s in states])
    ax.set_xticks(range(1, len(states) + 1), states, rotation=45)
    ax.set_title(ct, fontsize=9)
axes[0].set_ylabel("% of sample's cells"); plt.tight_layout(); plt.show()

## 7 · SLE subtypes (endotypes)

In [ ]:
from projects.pbmc_sle_classification.endotypes import profiles, healthy_centre, choose_k, describe
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

E = cfg["endotypes"]
F = healthy_centre(profiles(adata, samples, D["sample_key"], E["ifn_genes"]), samples)
sle = samples["sle"] == 1
X = PCA(n_components=min(10, F.shape[1]), random_state=0).fit_transform(F[sle].to_numpy())
k, sel = choose_k(X, E["k_range"], E["min_size"], E["n_boot"], cfg["train"]["seed"])
k = k if E["k"] == "auto" else int(E["k"])
display(sel.round(3))
lab = KMeans(k, n_init=50, random_state=0).fit_predict(X)
prof = F[sle].groupby(lab).mean()
for c in prof.index:   # named by what sets each cluster apart from the other SLE samples
    print(f"cluster {c}: n={int((lab == c).sum())}  {describe(prof.loc[c] - F[sle].mean())}")
display(pd.crosstab(lab, samples.loc[sle, "disease_state"].values, rownames=["cluster"]))
display(prof.T.style.background_gradient(cmap="RdBu_r", axis=None, vmin=-2, vmax=2).format("{:.2f}"))

In [ ]:
# Use these clusters as class labels (same ordering and naming as endotypes.py: by interferon level, SLE-A first)
ifn_cols = [c for c in prof.columns if c.startswith("IFN")]
order = prof[ifn_cols].mean(axis=1).sort_values(ascending=False).index
rename = {old: f"SLE-{'ABCDEFGH'[i]}" for i, old in enumerate(order)}
samples["endotype"] = "healthy"
samples.loc[sle, "endotype"] = pd.Series(lab, index=samples.index[sle]).map(rename)
classes = ["healthy"] + [rename[o] for o in order]
samples["class"] = samples["endotype"].map({c: i for i, c in enumerate(classes)})
samples["endotype"].value_counts()

## 8 · Train and score (one CV fold is quick; all folds for the real number)

In [ ]:
import torch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score
from projects.pbmc_sle_classification.features import baseline_features, composition, to_dense
from projects.pbmc_sle_classification.train import feature_spec, make_baseline, proba_full, train_mil, predict_mil, score

genes = adata.var_names[adata.var["highly_variable"]].tolist()
counts = adata[:, genes].layers["counts"]
sid = adata.obs[D["sample_key"]].astype(str).values
groups = pd.Series(np.arange(len(sid))).groupby(sid).indices
ids = samples.index.to_numpy()
ct = adata.obs["cell_type_fine"].astype(str).to_numpy()
Z = np.asarray(adata.obsm["X_scVI"], dtype=np.float32)
spec = feature_spec(adata, counts, D["sample_key"], cfg["train"])
Xb = np.stack([baseline_features(to_dense(counts[groups[s]]), ct[groups[s]], spec) for s in ids])
extras = np.stack([composition(ct[groups[s]], spec["cell_types"]) for s in ids])
bags = [Z[groups[s]] for s in ids]
y, sle_y, donors, K = samples["class"].to_numpy(int), samples["sle"].to_numpy(int), samples["donor"].astype(str).to_numpy(), len(classes)
folds = list(StratifiedGroupKFold(n_splits=cfg["train"]["n_splits"], shuffle=True, random_state=0).split(ids, y, donors))
print(f"baseline features {Xb.shape}, MIL bags {len(bags)}, classes {classes}")

In [ ]:
N_FOLDS = 1     # 1 for a quick look while tuning, cfg["train"]["n_splits"] for the full CV
oof = {m: np.full((len(y), K), np.nan) for m in ("baseline", "mil")}
for tr, te in folds[:N_FOLDS]:
    base = make_baseline(cfg["train"]).fit(Xb[tr], y[tr])
    oof["baseline"][te] = proba_full(base, Xb[te], K)
    net = train_mil(bags, extras, y, tr, K, cfg, seed=0)
    oof["mil"][te] = np.stack([predict_mil(net, bags[i], extras[i]) for i in te])
done = ~np.isnan(oof["mil"][:, 0])
oof["ensemble"] = (oof["baseline"] + oof["mil"]) / 2
for m, p in oof.items():
    print(m, "SLE AUROC %.3f" % roc_auc_score(sle_y[done], 1 - p[done, 0]),
          "| macro AUROC %.3f" % roc_auc_score(y[done], p[done], multi_class="ovr", labels=range(K)) if len(set(y[done])) == K else "")
ConfusionMatrixDisplay.from_predictions(y[done], oof["ensemble"][done].argmax(1), display_labels=classes, xticks_rotation=45)
plt.show()

In [ ]:
# Which cells does the MIL look at? (last trained fold's model, its test samples)
from shared.plotting import attention_by_group
rows = {}
for i in te:
    _, a = net(torch.tensor(bags[i]), torch.tensor(extras[i]) if net.d_extra else None)
    rows[ids[i]] = attention_by_group(a.detach().numpy(), ct[groups[ids[i]]])
att = pd.DataFrame(rows).T.fillna(0).groupby(samples.loc[ids[te], "endotype"].values).mean().T
display((att * 100).round(1).sort_values(att.columns[-1], ascending=False).head(12))

## 9 · Keep what worked

In [ ]:
# Paste into projects/pbmc_sle_classification/configs/default.yaml, commit, then train the final model with train_colab.ipynb
print(yaml.safe_dump({k: cfg[k] for k in ("qc", "represent", "endotypes", "model", "train")}, sort_keys=False))